### https://www.kaggle.com/competitions/drawing-with-llms

In [1]:
import kagglehub
import pandas as pd

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = ( None )
load_in_4bit = False 

#unsloth/Llama-3.2-3B-Instruct
#Qwen/Qwen2.5-Coder-3B
#Qwen/Qwen2.5-Coder-7B-Instruct
#Qwen/Qwen2.5-14B-Instruct


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-14B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=True,
    load_in_8bit=False
)
print(model.dtype)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.3.18: Fast Qwen2 patching. Transformers: 4.49.0.
   \\   /|    NVIDIA GeForce RTX 4070 Ti SUPER. Num GPUs = 1. Max memory: 15.695 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/202k [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

In [6]:
model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048, padding_idx=151665)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
      )
 

In [7]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 128, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

In [4]:
df=pd.read_csv('svg_score_train1.csv')
df=df[df['score'] > 0.5]
df=df[['topic','svg_code']]

In [5]:
df.shape

(1062, 2)

In [6]:
df1=pd.read_csv('svg_score_first100k.csv')
df1=df1[df1['base_score'] > 0.8]
df1=df1[['caption_blip2','clean_svg_code']]
df1.columns=['topic','svg_code']
print(df1.shape)

(5972, 2)


In [7]:
# df=pd.concat([df,df1.iloc[:3000]],ignore_index=True)
# print(df.shape)

In [8]:
df

,topic,svg_code
0,"'Golden wheat fields under a setting sun',","<svg viewBox=""0 0 200 200"" width=""200"" height=..."
3,"'Snowy mountains under a clear blue sky',","<svg viewBox=""0 0 200 100"" width=""200"" height=..."
4,'Checkerboard pattern with alternating green ...,"<svg viewBox=""0 0 100 100"" width=""100"" height=..."
6,"'Crimson and gold spirals intertwining',","<svg viewBox=""0 0 200 200"" width=""200"" height=..."
7,"'A navy blue trench coat with brass buttons',","<svg viewBox=""0 0 200 300"" width=""200"" height=..."
...,...,...
3075,A quiet wheat field with a scarecrow,"<svg viewBox=""0 0 200 150"" width=""400"" height=..."
3080,A quiet copse,"<svg viewBox=""0 0 200 150"" width=""400"" height=..."
3085,A sleek black tuxedo with a bow tie,"<svg viewBox=""0 0 200 300"" width=""200"" height=..."
3088,A smooth gradient of soft blues,"<svg width=""200"" height=""100"" viewBox=""0 0 200..."


In [9]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token  # Must add EOS_TOKEN

def formatting_prompts_func(examples):
    topics = examples["topic"]  # Using 'topic' as instruction
    svgs = examples["svg_code"]  # Using 'svg_code' as output
    texts = []
    
    for topic, svg_code in zip(topics, svgs):
        # No additional input is needed, so we pass an empty string
        text = alpaca_prompt.format(f"Generate an SVG image for the topic:",topic, svg_code) + EOS_TOKEN
        texts.append(text)
    
    return { "text": texts }

from datasets import Dataset
import pandas as pd
# Convert DataFrame to Hugging Face Dataset
dataset = Dataset.from_pandas(df)

# Apply formatting function
dataset = dataset.map(formatting_prompts_func, batched=True)


Map:   0%|          | 0/1062 [00:00<?, ? examples/s]

In [10]:
# Check dataset sample output
print(dataset["text"][0])

Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Generate an SVG image for the topic:

### Input:
'Golden wheat fields under a setting sun',

### Response:
<svg viewBox="0 0 200 200" width="200" height="200" xmlns="http://www.w3.org/2000/svg">
  <!-- Background for the sky -->
  <rect x="0" y="0" width="200" height="100" fill="orange" opacity="0.7"/>
  
  <!-- Sun -->
  <circle cx="100" cy="50" r="30" fill="yellow" opacity="0.8"/>
  
  <!-- Wheat fields -->
  <rect x="0" y="100" width="200" height="100" fill="goldenrod"/>
  
  <!-- Wheat stalks -->
  <g stroke="saddlebrown" stroke-width="2">
    <line x1="30" y1="100" x2="30" y2="150"/>
    <line x1="50" y1="100" x2="50" y2="150"/>
    <line x1="70" y1="100" x2="70" y2="150"/>
    <line x1="90" y1="100" x2="90" y2="150"/>
    <line x1="110" y1="100" x2="110" y2="150"/>
    <line x1="130" y1="100" x2="130" y2="1

In [11]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset=test_dataset,  # Add test dataset here
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 5, 
        max_steps = 1000,
        learning_rate = 1e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit", # "adamw_torch" better for fp16
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 123,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

Converting train dataset to ChatML (num_proc=2):   0%|          | 0/1062 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=2):   0%|          | 0/1062 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=2):   0%|          | 0/1062 [00:00<?, ? examples/s]

Truncating train dataset (num_proc=2):   0%|          | 0/1062 [00:00<?, ? examples/s]

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 1,062 | Num Epochs = 8
O^O/ \_/ \    Batch size per device = 2 | Gradient Accumulation steps = 4
\        /    Total batch size = 8 | Total steps = 1,000
 "-____-"     Number of trainable parameters = 161,480,704


Step,Training Loss
5,0.482900
10,0.459100
15,0.346500
20,0.301600
25,0.265800
30,0.270000
35,0.269700
40,0.222100
45,0.226400
50,0.229500


In [ ]:
# #This ONLY saves the LoRA adapters, and not the full model.
# model.save_pretrained("./lora/lora_model_3b_v3") # Local saving
# tokenizer.save_pretrained("./lora/lora_model_3b_v3")

In [ ]:
# import gc
# del model
# gc.collect()
# torch.cuda.empty_cache()

In [ ]:
# import psutil

# def kill_large_python_processes():
#     # Loop over all running processes
#     for proc in psutil.process_iter(['pid', 'name', 'memory_info', 'exe']):
#         try:
#             # Check if the process is Python and type is 'C' (for computation)
#             if 'python' in proc.info['name'].lower():
#                 # Check if memory usage is greater than 2048 MB
#                 memory_usage_mb = proc.info['memory_info'].rss / (1024 * 1024)  # Convert bytes to MB
#                 if memory_usage_mb > 2048:
#                     print(f"Killing Python process with PID {proc.info['pid']} using {memory_usage_mb} MB memory")
#                     proc.kill()  # Kill the process
#         except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess):
#             pass  # Handle processes that might disappear during iteration

# #kill_large_python_processes()

In [ ]:
# from unsloth import FastLanguageModel
# model, tokenizer = FastLanguageModel.from_pretrained(
# model_name = "./lora/lora_model_3b_v2", # YOUR MODEL YOU USED FOR TRAINING
# max_seq_length = 2048,
# dtype = (None),
# load_in_4bit = False,
# )
# FastLanguageModel.for_inference(model) # Enable native 2x faster inference


In [ ]:
# # alpaca_prompt = You MUST copy from above!
# alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

# ### Instruction:
# {}

# ### Input:
# {}

# ### Response:
# {}"""

# inputs = tokenizer(
# [
#     alpaca_prompt.format(
#         "Please write a SVG code fo rthe given topic?", # instruction
#         "Golden sun rising in the east", # input
#         "", # output - leave this blank for generation!
#     )
# ], return_tensors = "pt").to("cuda")

# outputs = model.generate(**inputs, max_new_tokens = 1024, use_cache = True)
# tokenizer.batch_decode(outputs)

In [ ]:
#save merged 16bit
model.save_pretrained_merged("./lora/lora_qwen_3B_v1", tokenizer, save_method = "merged_16bit",)